# 02 — Geodata Exploration & Cleaning 地理資料檢視與清理

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewwangarchnycu/gis-open-data-workshop-2026/blob/main/notebooks/02_geodata_exploration.ipynb)

Part of [Mapping the Unknown](https://github.com/andrewwangarchnycu/gis-open-data-workshop-2026) — see [Lesson 02](../lessons/02-space-to-data/) and [Lesson 05](../lessons/05-computational-gis/).

**Spatial meaning first 先談空間意義**: before analyzing any spatial dataset, you must inspect it — what geometry type, what attributes, what CRS — and clean it. Skipping this step is the #1 cause of wrong results later. 在分析任何空間資料前，必須先檢視其幾何類型、屬性與 CRS，並進行清理。跳過此步驟是後續分析結果出錯的頭號原因。

In [ ]:
!pip install geopandas shapely matplotlib contextily -q

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon
import matplotlib.pyplot as plt
import contextily as cx

## Step 1 — Load (self-contained sample, same as Notebook 01) 載入（自足範例，與筆記本 01 相同）

This notebook is self-contained — no local files required — so we rebuild the sample data here, plus a deliberately messy version to practice cleaning on.
本筆記本自成一體，不需任何本機檔案，因此我們在此重建範例資料，並刻意加入不完整版本以練習清理。

In [ ]:
tree_coords = [
    (121.5320, 25.0335), (121.5325, 25.0338), (121.5330, 25.0330),
    (121.5340, 25.0345), (121.5345, 25.0348), (121.5300, 25.0320),
    (121.5305, 25.0322), (121.5360, 25.0360), (None, None),  # a bad row on purpose 刻意放入的錯誤資料
]
geoms = [Point(xy) if xy[0] is not None else None for xy in tree_coords]
trees = gpd.GeoDataFrame(
    {"tree_id": range(1, len(tree_coords) + 1), "species": ["unknown"] * (len(tree_coords) - 1) + [None]},
    geometry=geoms,
    crs="EPSG:4326",
)

public_spaces = gpd.GeoDataFrame(
    {"name": ["Riverside Park", "Central Plaza", "Corner Lot Square"]},
    geometry=[
        Polygon([(121.5315, 25.0328), (121.5335, 25.0328), (121.5335, 25.0350), (121.5315, 25.0350)]),
        Polygon([(121.5295, 25.0315), (121.5310, 25.0315), (121.5310, 25.0328), (121.5295, 25.0328)]),
        Polygon([(121.5352, 25.0352), (121.5365, 25.0352), (121.5365, 25.0365), (121.5352, 25.0365)]),
    ],
    crs="EPSG:4326",
)

# A layer loaded with the WRONG crs, on purpose, to demonstrate the #1 beginner bug
# 刻意以錯誤 CRS 載入的圖層，用來示範初學者最常見的錯誤
roads_wrong_crs = gpd.GeoDataFrame(
    {"name": ["Riverside Ave"]},
    geometry=[Polygon([(121.531, 25.032), (121.534, 25.032), (121.534, 25.033), (121.531, 25.033)]).exterior],
    crs="EPSG:3826",  # WRONG on purpose — coordinates are actually lon/lat (4326)
)

trees

## Step 2 — Inspect 檢視
Always check: row count, geometry type, columns, CRS, and missing values — before doing anything else.
務必先檢查：列數、幾何類型、欄位、CRS 與缺失值——再進行其他操作。

In [ ]:
print("Rows:", len(trees))
print("Geometry types:", trees.geom_type.unique())
print("Columns:", list(trees.columns))
print("CRS:", trees.crs)
print("Missing geometry:", trees.geometry.isna().sum())
print("Missing species:", trees["species"].isna().sum())

**Expected output 預期輸出**: `Rows: 9`, one missing geometry, one missing species — this is the bad row we added on purpose.
`Rows: 9`，一筆缺失幾何、一筆缺失樹種——這是我們刻意加入的錯誤資料列。

## Step 3 — Clean 清理
Drop rows with missing/invalid geometry — a dataset with `None` geometry will silently break spatial operations later.
刪除缺失／無效幾何的資料列——含有 `None` 幾何的資料集會在後續空間運算中悄悄出錯。

In [ ]:
trees_clean = trees[trees.geometry.notnull()].copy()
trees_clean["species"] = trees_clean["species"].fillna("unrecorded")
print("Before 清理前:", len(trees), "→ After 清理後:", len(trees_clean))

## Step 4 — Fix a CRS mismatch CRS 不一致的修正
`roads_wrong_crs` was loaded declaring EPSG:3826 (a projected, meter-based CRS), but the coordinates are actually longitude/latitude (EPSG:4326). Overlaying it with `trees_clean` right now would place it in completely the wrong location on Earth.
`roads_wrong_crs` 宣告為 EPSG:3826（投影、公尺制 CRS），但座標實際上是經緯度（EPSG:4326）。若現在直接與 `trees_clean` 疊合，位置會完全錯誤。

In [ ]:
# The fix: tell geopandas the TRUE crs of the raw coordinates (do not reproject — relabel)
# 修正方式：告訴 geopandas 原始座標「真正」的 CRS（不是重新投影，而是重新標記）
roads_fixed = roads_wrong_crs.set_crs("EPSG:4326", allow_override=True)
print("Fixed CRS:", roads_fixed.crs)

## Step 5 — Add a basemap 加入底圖

**Why 為什麼**: a plot of colored shapes on white space doesn't read as a place. Adding real street/building tiles underneath — via `contextily` — turns the same data into something a reader immediately recognizes as an actual location.
在空白背景上畫幾個色塊，看起來不像一個地方。用 `contextily` 在圖層下方加入真實街道／建築底圖，能讓同一份資料立刻讓讀者辨識出這是一個真實地點。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
public_spaces.plot(ax=ax, color="#c7e9c0", edgecolor="black", alpha=0.7)
roads_fixed.plot(ax=ax, color="grey", linewidth=2)
trees_clean.plot(ax=ax, color="#238b45", markersize=30)

# Basemap tiles, reprojected to match our layers' CRS (EPSG:4326)
# 底圖圖磚，重新投影至與圖層相同的 CRS（EPSG:4326）
cx.add_basemap(ax, crs=public_spaces.crs.to_string(), source=cx.providers.CartoDB.Positron)

ax.set_title("Cleaned layers, aligned CRS 清理後圖層，CRS 已對齊")
ax.set_axis_off()
plt.show()

## Research interpretation exercise 研究詮釋練習

1. What would have gone wrong downstream if you had skipped the CRS check? 若跳過 CRS 檢查，後續會出什麼錯？
2. Real open datasets often have missing values — how would you decide whether to drop or fill them for *your* research question? 真實開放資料常有缺失值——針對你的研究問題，你會如何決定刪除或填補？
3. Continue to [`03_spatial_analysis.ipynb`](03_spatial_analysis.ipynb) to run buffer, join, and intersection operations.